# From one sentence to reusable plotting code

A short story in two datasets: ask for a chart, refine it in plain language, inspect the implementation, replay it without another model call, and reuse it on new data. Both examples use real sample datasets loaded through Augplot from Seaborn's public catalog.

**Setup:** install using the [README](../README.md#install), select your Augplot kernel, and run the first cell. It selects `openai/gpt-5.6-terra` and asks for your API key with hidden input.

First run: four generations; each may need one repair request. Identical reruns use saved code. Loading a dataset needs internet access the first time; Seaborn caches it locally. Your provider receives a data profile, and generated Python runs locally without a security sandbox.

In [ ]:
import os
from getpass import getpass

os.environ["AUGPLOT_MODEL"] = "openai/gpt-5.6-terra"
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

In [ ]:
import augplot as ap

## 1. Ask for a useful first view

The Palmer penguins dataset has body and bill measurements for three species across three islands. Load it without a separate Seaborn import, then ask Augplot for a useful first view.

The next cell adds facets, trends, and another visual encoding while keeping the prompt short.

In [ ]:
penguin_prompt = "Compare bill length and depth across penguin species."
penguins = ap.load_sns_dataset("penguins")
plt = ap.plot(
    penguins,
    backend="seaborn",
    prompt=penguin_prompt,
)

In [ ]:
print(plt.explanation)
print("Reused saved code:", plt.cache_hit)

## 2. Refine the instruction, not the plotting code

This refinement coordinates small multiples, per-species trend lines, consistent axes, and marker shapes for sex—details that otherwise require several plotting calls and legend cleanup. A regression line computed by the selected visualization backend is in scope because it exists only as a layer in the rendered figure.

In [ ]:
penguin_refinement = (
    "Use one panel per species with shared axes, add a Seaborn regression trend "
    "as a visual summary in each panel, and keep sex visible with marker shape."
)
plt.refine(penguin_refinement)

## 3. Inspect what one sentence produced

The chart is only half the result. Augplot also keeps the complete function available for inspection, reuse, and normal version control.

In [ ]:
# Summarize the result, then display the generated function.
from IPython.display import Code

line_count = len(plt.code.strip().splitlines())
source_origin = "a history replay" if plt.cache_hit else "one refinement sentence"
print(
    f"Result of {source_origin}: a {line_count}-line reusable Python function."
)
print(
    "It coordinates faceting, shared axes, sex encoding, per-panel trends, "
    "labels, legends, and layout."
)
print("Saved history source:", plt.history_path)
Code(plt.code, language="python")

## 4. Prove that the work persists

Start again with the same data and the same two instructions. Both steps replay from `.augplot/plots` without another model request, even though this is a new chart object. See [how visualization history works](../docs/visualization-history.md) for cache keys, branches, and regeneration.

In [ ]:
replayed = ap.plot(
    penguins,
    backend="seaborn",
    prompt=penguin_prompt,
    show=False,
)
initial_replayed = replayed.cache_hit
replayed.refine(penguin_refinement, show=False)

print("Original request replayed:", initial_replayed)
print("Refinement replayed:", replayed.cache_hit)
print("Same reusable function:", replayed.code == plt.code)

## 5. Reuse the function on a real subset

Use only penguins observed on Biscoe Island to show that `render()` can apply the generated function to compatible data without another model call. Write the function as reusable Python under a readable name when you're happy with it.

In [ ]:
biscoe_penguins = penguins[penguins["island"] == "Biscoe"]
plt.render(biscoe_penguins, title="Penguin bills on Biscoe Island")

## 6. Save the chart as ordinary Python

The implementation is not trapped inside Augplot. `to_python()` writes the current chart to `augplot_utils.py`; new chart functions are appended, and rerunning the same one updates its generated definition. The resulting module needs neither Augplot nor provider credentials.

In [ ]:
plt.to_python(function_name="plot_penguin_bills")

In [ ]:
from augplot_utils import plot_penguin_bills

fig = plot_penguin_bills(biscoe_penguins, figsize=(11, 5))
fig

## 7. Change the question: reveal seasonality

The flights dataset contains monthly passenger totals from 1949 through 1960. Start with a line chart so the overall growth and seasonal cycles are visible.

Because Seaborn assigns an ordered categorical type to `month`, the calendar order is available in the data profile.

In [ ]:
flights = ap.load_sns_dataset("flights")
flight_viz = ap.plot(
    flights,
    backend="seaborn",
    prompt="Plot monthly airline passengers over time, with one line per year.",
)

## 8. Reframe the same data as a seasonal heatmap

A heatmap makes both long-term growth and recurring seasonality easier to scan. The refinement must pivot the data, preserve calendar order, annotate cells, and identify the peak.

In [ ]:
flight_viz.refine(
    "Turn this into a year-by-month heatmap, preserve calendar month order, "
    "annotate each cell, and emphasize the busiest month."
)

**Where next:** repeating only `refine()` edits the current version again and creates a new branch. Keep `.augplot/` with the notebook when you want its generated history to travel with it. Read [how visualization history works](../docs/visualization-history.md).